# MergeSortBottomUp (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/MergeSortBottomUp_python.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `MergeSortBottomUp`: the algorithm in Python, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/MergeSortBottomUp_cpp.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/MergeSortBottomUp_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_python.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Python 3, nothing to install.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `fib_rec` (snake_case).

## Recursive MergeSort and MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: an array $a[1..n]$ of natural numbers. *Output*: the array sorted in non-decreasing order. The same splitting as Merge Sort, but **without recursion**: first we merge pairs of sub-arrays of length $1$, then of length $2$, $4$, … with the procedure `Merge` of chapter 17 (repeated here). It executes $\lceil\log_2 n\rceil$ passes and $\Theta(n\log_2 n)$ steps, like the recursive version, and does not use the call stack.

First the example of the chapter $a=[5,2,4,6,1,3]$, pass by pass (indices are from 0). Then the comparison, on pseudo-random arrays (same generator in all languages, seed $12345$, values between $1$ and $10^6$), of the number of **comparisons** between elements of `InsertionSort`, `MergeSort` and `MergeSortBottomUp`: $\Theta(n^2)$ against $\Theta(n\log_2 n)$, with the two versions of Merge Sort practically equal ("OK" means that the three algorithms give the same sorted array), and the measured times.

In [1]:
import time

comparisons = 0   # comparisons between elements

def merge(a, l, m, r):
    """Merge(a, l, m, r) of chapter 17."""
    global comparisons
    n1 = m - l + 1
    n2 = r - m
    L = [0] * n1
    R = [0] * n2
    for i in range(n1):
        L[i] = a[l + i]
    for j in range(n2):
        R[j] = a[m + 1 + j]
    i = 0
    j = 0
    k = l
    while i < n1 and j < n2:
        comparisons += 1
        if L[i] <= R[j]:
            a[k] = L[i]
            i += 1
        else:
            a[k] = R[j]
            j += 1
        k += 1
    while i < n1:
        a[k] = L[i]
        i += 1
        k += 1
    while j < n2:
        a[k] = R[j]
        j += 1
        k += 1

def merge_sort(a, l, r):
    if l < r:
        m = l + (r - l) // 2
        merge_sort(a, l, m)
        merge_sort(a, m + 1, r)
        merge(a, l, m, r)

def merge_sort_bottom_up(a, n):
    """MergeSortBottomUp(a, n): sorts a[0..n-1] without recursion."""
    width = 1
    while width < n:
        l = 0
        while l + width < n:                    # pseudo-code: l + width <= n
            m = l + width - 1
            r = min(l + 2 * width - 1, n - 1)
            merge(a, l, m, r)
            l = l + 2 * width
        width = 2 * width

def merge_sort_bottom_up_trace(a, n):
    width = 1
    passes = 0
    while width < n:
        l = 0
        parts = []
        while l + width < n:
            m = l + width - 1
            r = min(l + 2 * width - 1, n - 1)
            merge(a, l, m, r)
            parts.append(f"a[{l}..{m}] with a[{m + 1}..{r}]")
            l = l + 2 * width
        passes += 1
        print(f"width={width}: merge " + ", ".join(parts) + f" => {a}")
        width = 2 * width
    print(f"passes = {passes}")

def insertion_sort(a, n):
    """InsertionSort(a, n) (chapter 3), with the count of comparisons."""
    global comparisons
    for j in range(1, n):
        key = a[j]
        i = j - 1
        while i >= 0:
            comparisons += 1
            if a[i] > key:
                a[i + 1] = a[i]
                i = i - 1
            else:
                break
        a[i + 1] = key

def random_array(n, seed=12345):
    """Linear congruential generator: same numbers in Python, C++ and Java."""
    x = seed
    a = []
    for _ in range(n):
        x = (x * 1103515245 + 12345) % 2147483648
        a.append(1 + x % 1000000)
    return a

a = [5, 2, 4, 6, 1, 3]
print("a =", a, "  n =", len(a))
merge_sort_bottom_up_trace(a, len(a))

print()
print(f"{'n':>6s} {'InsertionSort':>14s} {'MergeSort':>10s} {'BottomUp':>9s}  comparisons")
times = []
for n in [1000, 2000, 4000]:
    base = random_array(n)
    x1, x2, x3 = base[:], base[:], base[:]
    comparisons = 0
    t0 = time.perf_counter(); insertion_sort(x1, n); t1 = (time.perf_counter() - t0) * 1000
    c1 = comparisons
    comparisons = 0
    t0 = time.perf_counter(); merge_sort(x2, 0, n - 1); t2 = (time.perf_counter() - t0) * 1000
    c2 = comparisons
    comparisons = 0
    t0 = time.perf_counter(); merge_sort_bottom_up(x3, n); t3 = (time.perf_counter() - t0) * 1000
    c3 = comparisons
    ok = "OK" if (x1 == x2 == x3 == sorted(base)) else "ERROR"
    print(f"{n:>6d} {c1:>14d} {c2:>10d} {c3:>9d}  {ok}")
    times.append((n, t1, t2, t3))
print()
print("Measured times (they depend on the machine):")
for n, t1, t2, t3 in times:
    print(f"n={n:<6d} InsertionSort {t1:>10.3f} ms   MergeSort {t2:>8.3f} ms   MergeSortBottomUp {t3:>8.3f} ms")

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] with a[1..1], a[2..2] with a[3..3], a[4..4] with a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] with a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] with a[4..5] => [1, 2, 3, 4, 5, 6]
passes = 3

     n  InsertionSort  MergeSort  BottomUp  comparisons
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Measured times (they depend on the machine):
n=1000   InsertionSort     24.227 ms   MergeSort    2.120 ms   MergeSortBottomUp    2.154 ms
n=2000   InsertionSort    101.923 ms   MergeSort    4.609 ms   MergeSortBottomUp    4.690 ms
n=4000   InsertionSort    410.813 ms   MergeSort   10.007 ms   MergeSortBottomUp   10.087 ms


**Try it yourself.** Change $n$ (with $n=8000$ the comparisons of `InsertionSort` quadruple with respect to $n=4000$, those of Merge Sort little more than double).

In [2]:
# Try n = 8000: the comparisons of InsertionSort quadruple with respect to n = 4000, those of Merge Sort little more than double.
n = 8000
base = random_array(n)
x1, x2 = base[:], base[:]
comparisons = 0
insertion_sort(x1, n)
c1 = comparisons
comparisons = 0
merge_sort_bottom_up(x2, n)
print(c1, comparisons, x1 == x2)

16115652 93756 True


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).